In [0]:
journey = spark.table(
    "airline_catalog.gold.passenger_journey_gold"
)

In [0]:
journey.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "airline_catalog.gold.journey_delta"
    )

In [0]:
%sql
SELECT * FROM airline_catalog.gold.journey_delta;

passenger_name,flight_id,booking_id,travel_class,ticket_price,booking_date,revenue,price_band,airline,from_city,to_city,duration,status,delay_flag,meal,seat
Rahul Sharma,F101,B1001,Economy,8500,2026-06-01,8500,Budget,Indigo,Hyderabad,Delhi,140,On Time,No,Veg,Window
Priya Reddy,F101,B1002,Business,22000,2026-06-01,22000,Premium,Indigo,Hyderabad,Delhi,140,On Time,No,Non-Veg,Aisle
Amit Kumar,F102,B1003,Economy,9000,2026-06-02,9000,Budget,Air India,Mumbai,Chennai,120,Delayed,Yes,null,null
Sneha Patel,F103,B1004,Premium Economy,15000,2026-06-02,15000,Standard,Vistara,Bangalore,Hyderabad,90,On Time,No,null,null
Farhan Ali,F104,B1005,Economy,7500,2026-06-03,7500,Budget,Indigo,Delhi,Mumbai,130,Cancelled,No,null,null
Neha Singh,F105,B1006,Business,25000,2026-06-03,25000,Premium,Air India,Chennai,Bangalore,80,On Time,No,null,null
Arjun Verma,F106,B1007,Economy,10000,2026-06-04,10000,Budget,Akasa,Pune,Delhi,150,Delayed,Yes,null,null
Meera Nair,F107,B1008,Premium Economy,17000,2026-06-04,17000,Standard,Vistara,Hyderabad,Kolkata,160,On Time,No,null,null
Kiran Rao,F108,B1009,Economy,9500,2026-06-05,9500,Budget,Indigo,Mumbai,Hyderabad,110,On Time,No,null,null
Nisha Reddy,F109,B1010,Business,28000,2026-06-05,28000,Premium,Akasa,Delhi,Chennai,145,Delayed,Yes,null,null


In [0]:
bookings = spark.table(
    "airline_catalog.silver.bookings_silver"
)

bookings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "airline_catalog.silver.booking_master"
    )

In [0]:
from pyspark.sql import Row

updates = [
    Row(
        booking_id="B1001",
        flight_id="F101",
        passenger_name="Rahul Sharma",
        travel_class="Business",
        ticket_price=25000,
        booking_date="2026-06-02",
        revenue=25000,
        price_band="Premium"
    ),

    Row(
        booking_id="B2001",
        flight_id="F102",
        passenger_name="Amit Kumar",
        travel_class="Economy",
        ticket_price=9000,
        booking_date="2026-06-02",
        revenue=9000,
        price_band="Budget"
    )
]

updates_df = spark.createDataFrame(updates)

display(updates_df)

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1001,F101,Rahul Sharma,Business,25000,2026-06-02,25000,Premium
B2001,F102,Amit Kumar,Economy,9000,2026-06-02,9000,Budget


In [0]:
updates_df.createOrReplaceTempView(
    "updates_view"
)

SCD Type 1

In [0]:
%sql
MERGE INTO airline_catalog.silver.booking_master tgt
USING updates_view src

ON tgt.booking_id = src.booking_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT *
FROM airline_catalog.silver.booking_master;

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1002,F101,Priya Reddy,Business,22000,2026-06-01,22000,Premium
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,9000,Budget
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,15000,Standard
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,7500,Budget
B1006,F105,Neha Singh,Business,25000,2026-06-03,25000,Premium
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,10000,Budget
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,17000,Standard
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,9500,Budget
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,28000,Premium
B1011,F110,David Thomas,Economy,8000,2026-06-06,8000,Budget


Time Travel

In [0]:
%sql
DESCRIBE HISTORY airline_catalog.silver.booking_master;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-06-20T05:42:58.000Z,143016486184747,atsthasneem@gmail.com,MERGE,"Map(predicate -> [""(booking_id#16387 = booking_id#16370)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4304045627377424),3bebb5f5-f6b4-4a9d-8a40-b3ccc50e64ac,0620-042819-i8u41su3-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4759, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6730, materializeSourceTimeMs -> 212, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3587, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2790)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-20T05:41:50.000Z,143016486184747,atsthasneem@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4304045627377424),b5527acf-0031-463d-8f8d-7d80043fe6fd,0620-042819-i8u41su3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 3146)",null,Databricks-Runtime/18.2.x-photon-scala2.13


In [0]:
%sql
SELECT *
FROM airline_catalog.silver.booking_master
VERSION AS OF 0;

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1001,F101,Rahul Sharma,Economy,8500,2026-06-01,8500,Budget
B1002,F101,Priya Reddy,Business,22000,2026-06-01,22000,Premium
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,9000,Budget
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,15000,Standard
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,7500,Budget
B1006,F105,Neha Singh,Business,25000,2026-06-03,25000,Premium
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,10000,Budget
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,17000,Standard
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,9500,Budget
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,28000,Premium


In [0]:
%sql
SELECT *
FROM airline_catalog.silver.booking_master
VERSION AS OF 1;

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1002,F101,Priya Reddy,Business,22000,2026-06-01,22000,Premium
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,9000,Budget
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,15000,Standard
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,7500,Budget
B1006,F105,Neha Singh,Business,25000,2026-06-03,25000,Premium
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,10000,Budget
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,17000,Standard
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,9500,Budget
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,28000,Premium
B1011,F110,David Thomas,Economy,8000,2026-06-06,8000,Budget


In [0]:
%sql
SELECT *
FROM airline_catalog.silver.booking_master;

booking_id,flight_id,passenger_name,travel_class,ticket_price,booking_date,revenue,price_band
B1002,F101,Priya Reddy,Business,22000,2026-06-01,22000,Premium
B1003,F102,Amit Kumar,Economy,9000,2026-06-02,9000,Budget
B1004,F103,Sneha Patel,Premium Economy,15000,2026-06-02,15000,Standard
B1005,F104,Farhan Ali,Economy,7500,2026-06-03,7500,Budget
B1006,F105,Neha Singh,Business,25000,2026-06-03,25000,Premium
B1007,F106,Arjun Verma,Economy,10000,2026-06-04,10000,Budget
B1008,F107,Meera Nair,Premium Economy,17000,2026-06-04,17000,Standard
B1009,F108,Kiran Rao,Economy,9500,2026-06-05,9500,Budget
B1010,F109,Nisha Reddy,Business,28000,2026-06-05,28000,Premium
B1011,F110,David Thomas,Economy,8000,2026-06-06,8000,Budget


In [0]:
%sql
OPTIMIZE airline_catalog.silver.booking_master;

path,metrics
abfss://metastore@atsdatabricksunity.dfs.core.windows.net/9b7405bb-8507-45c6-b943-4779a4db7886/tables/cd7939fa-2b93-4133-a66a-4c0654914cff,"List(1, 3, List(3181, 3181, 3181.0, 1, 3181), List(2369, 3146, 2635.0, 3, 7905), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1781934442584, 1781934445786, 8, 1, null, List(1, 1), null, 8, 8, 1158, 0, null, null)"


In [0]:
%sql
OPTIMIZE airline_catalog.silver.booking_master
ZORDER BY (flight_id);

path,metrics
abfss://metastore@atsdatabricksunity.dfs.core.windows.net/9b7405bb-8507-45c6-b943-4779a4db7886/tables/cd7939fa-2b93-4133-a66a-4c0654914cff,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 3181), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781934489350, 1781934489896, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"


In [0]:
%sql
VACUUM airline_catalog.silver.booking_master;

path
abfss://metastore@atsdatabricksunity.dfs.core.windows.net/9b7405bb-8507-45c6-b943-4779a4db7886/tables/cd7939fa-2b93-4133-a66a-4c0654914cff


In [0]:
%sql 
DESCRIBE HISTORY airline_catalog.silver.booking_master;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-20T05:49:02.000Z,143016486184747,atsthasneem@gmail.com,VACUUM END,Map(status -> COMPLETED),null,List(4304045627377424),092007d6-ad79-4791-8a03-54f93d4be7e3,0620-042819-i8u41su3-v2n,3,SnapshotIsolation,true,"Map(numDeletedFiles -> 0, numVacuumedDirectories -> 1)",null,Databricks-Runtime/18.2.x-photon-scala2.13
3,2026-06-20T05:49:01.000Z,143016486184747,atsthasneem@gmail.com,VACUUM START,"Map(retentionCheckEnabled -> true, defaultRetentionMillis -> 604800000)",null,List(4304045627377424),092007d6-ad79-4791-8a03-54f93d4be7e3,0620-042819-i8u41su3-v2n,2,SnapshotIsolation,true,"Map(numFilesToDelete -> 0, sizeOfDataToDelete -> 0)",null,Databricks-Runtime/18.2.x-photon-scala2.13
2,2026-06-20T05:47:25.000Z,143016486184747,atsthasneem@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> false, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(4304045627377424),c7b71867-2e84-4b10-87c5-aa63e60dca72,0620-042819-i8u41su3-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7905, p25FileSize -> 3181, numDeletionVectorsRemoved -> 1, minFileSize -> 3181, numAddedFiles -> 1, maxFileSize -> 3181, p75FileSize -> 3181, p50FileSize -> 3181, numAddedBytes -> 3181)",null,Databricks-Runtime/18.2.x-photon-scala2.13
1,2026-06-20T05:42:58.000Z,143016486184747,atsthasneem@gmail.com,MERGE,"Map(predicate -> [""(booking_id#16387 = booking_id#16370)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(4304045627377424),3bebb5f5-f6b4-4a9d-8a40-b3ccc50e64ac,0620-042819-i8u41su3-v2n,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4759, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 6730, materializeSourceTimeMs -> 212, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 3587, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2790)",null,Databricks-Runtime/18.2.x-photon-scala2.13
0,2026-06-20T05:41:50.000Z,143016486184747,atsthasneem@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4304045627377424),b5527acf-0031-463d-8f8d-7d80043fe6fd,0620-042819-i8u41su3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 20, numOutputBytes -> 3146)",null,Databricks-Runtime/18.2.x-photon-scala2.13
